In [52]:
df

,text,label
8,Somebody mastered the difficult task of mergin...,1
4999,"""The Mother"" tells of a recently widowed mid-6...",1
10,This film takes you on one family's impossible...,1
4990,"There is some spectacular, heart stoppingly be...",1
4992,"If Jean Renoir's first film ""Whirlpool of Fate...",1
...,...,...
4973,"There are times when, less than halfway throug...",0
4974,I watched this because of the description and ...,0
4975,I'm going to spend as much time on this review...,0
4978,As one of the victims of the whole Enron scand...,0


In [19]:
import pandas as pd

In [20]:
df = pd.read_csv('movie_reviews.csv')

In [21]:
df.sort_values(by = 'label', ascending= False, inplace= True)

In [22]:
df_pos_100 = df.head(100)
df_neg_100 = df.tail(100)

In [23]:
pos_100 = df_pos_100.iloc[:,0].to_list()
neg_100 = df_neg_100.iloc[:,0].to_list()

In [24]:
neg_100[:10]

['If this is the best Commander Hamilton movie, I have no curiosity about the others.<br /><br />A movie actor\'s greatest tools are his eyes, but when Peter Stormare wants to show great emotion, he closes his, so for five or six seconds we get to admire his eyelids while his feelings remain unknown behind them. Lousy acting technique.<br /><br />Stormare also flinches sometimes when he fires a gun, turning his head away and clamping his eyes shut. Watch carefully. James Bond can rest easy with competition like this.<br /><br />There are some interesting supporting performances from other actors, but not enough to hang a whole movie on. The cinematography is good-looking, doing a fine job of capturing the Nordic cold. Even the Sahara winds up looking cold. Perhaps Hamilton carries his own climate with him.<br /><br />There are some individual good action sequences here. Unfortunately, the only sense of humor on screen belongs to the villain, which turns the hero into a big pill. James 

In [25]:
from nltk.corpus import stopwords
import string
english_stops = stopwords.words('english')

In [26]:
def my_clean_words(s):
    all_words = []
    review = s.translate(str.maketrans('','',string.punctuation))
    words = review.split()
    for w in words:
        w = w.lower()
        if w not in english_stops:
            all_words.append(w)
    return all_words

In [27]:
pos_100_words = []
pos_100_words_list = []

for review in pos_100:
    words_cleaned = my_clean_words(review)
    pos_100_words_list.append(words_cleaned)
    for word in words_cleaned:
        pos_100_words.append(word)

In [28]:
neg_100_words = []
neg_100_words_list = []

for review in neg_100:
    words_cleaned = my_clean_words(review)
    neg_100_words_list.append(words_cleaned)
    for word in words_cleaned:
        neg_100_words.append(word)

In [29]:
attribute_words = list(set(neg_100_words + pos_100_words))

In [30]:
# transform pos reviews into one-hot encodings

pos_vectors = []

for i in range(len(pos_100_words_list)):
    vector = []
    for j in range(len(attribute_words)):
        if attribute_words[j] in pos_100_words_list[i]:
            vector.append(1)
        else:
            vector.append(0)
    pos_vectors.append(vector)

In [31]:
# transform neg reviews into one-hot encodings

neg_vectors = []

for i in range(len(neg_100_words_list)):
    vector = []
    for j in range(len(attribute_words)):
        if attribute_words[j] in neg_100_words_list[i]:
            vector.append(1)
        else:
            vector.append(0)
    neg_vectors.append(vector)

In [32]:
import numpy as np
x = np.array(pos_vectors + neg_vectors)

In [33]:
pos_negative_labels = []
for i in range(100):
    pos_negative_labels.append(1)
for i in range(100):
    pos_negative_labels.append(0)

In [34]:
y = np.array(pos_negative_labels)

In [35]:
x.shape

(200, 8204)

In [36]:
from sklearn.naive_bayes import MultinomialNB
nb_classifier = MultinomialNB()
nb_classifier.fit(x,y)

MultinomialNB()

In [37]:
nb_classifier.predict(x[:10])

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [38]:
nb_classifier.predict(x[-10:])

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [40]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.5, random_state= 42)

In [41]:
NB_model = MultinomialNB()

In [42]:
NB_model = NB_model.fit(x_train, y_train)

In [43]:
y_train_pred = NB_model.predict(x_train)
y_test_pred = NB_model.predict(x_test)

In [44]:
print(accuracy_score(y_true=y_train,y_pred=y_train_pred))

1.0


In [45]:
print(accuracy_score(y_true=y_test,y_pred=y_test_pred))

0.72


In [46]:
from sklearn.model_selection import cross_val_score

In [48]:
scores = cross_val_score(NB_model, x, y, cv = 2)

In [49]:
scores

array([0.72, 0.7 ])

In [50]:
print('CV Scores: ', scores)
print('Avg CV Score: ', scores.mean())
print('# of CV Scores used in Avg: ', len(scores))

CV Scores:  [0.72 0.7 ]
Avg CV Score:  0.71
# of CV Scores used in Avg:  2


In [51]:
print(cross_val_score(NB_model, x, y, cv = 5))

[0.825 0.7   0.75  0.725 0.675]
